<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 37px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
  Machine Learning<br>
  </div> 

<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 35px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
  -<br>
  </div> 
  
<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 35px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
      Illustration with Random Forests<br><br>
  </div> 

<div style="font-variant: small-caps; 
      font-weight: normal; 
      font-size: 25px; 
      text-align: center; 
      padding: 15px; 
      margin: 10px;">
      <font color=orange> 3 - Regression</font>
  </div>

<a id="plan"></a>

## Regression with Random Forests

-  [Regression Model scoring, underfitting, overfitting](#model_scoring)<br>
-  [Base model](#model_base)<br>
-  [Decision Tree Regressor](#decision_tree)<br>
-  [Tree bagging](#tree_bagging)<br>
-  [OOB score](#oob_score)<br>
-  [Stoping criteria](#tree_pruning)<br>
-  [Data subsampling](#data_sampling)<br>
-  [Feature subsampling](#feature_sampling)<br>
-  [Bagging as general Ensemble method](#general_bagging)<br>
-  [Cross validation](#cross_validation)<br>
-  [Hyperparameter tuning for final model selection](#hyperparameter_tuning)<br>


### Imports

In [ ]:
# for package auto reload
%load_ext autoreload
%autoreload 2

# for better rendering of plots in jupyter notebook
%matplotlib inline

In [ ]:
# base modules
import logging
import math
from collections import OrderedDict
from pathlib import Path

import dill

# for manipulating data
import numpy as np
import pandas as pd

# for visualization
from matplotlib import pyplot as plt
from sklearn import metrics

# for Machine Learning
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor, plot_tree

In [ ]:
from eclyon.plot import draw_tree
from eclyon.transforms import process_df, split_vals

# Regression with Random Forests

[Back to top](#plan)

### $\bullet$ Regression Model scoring, underfitting, overfitting

[Back to top](#plan)

In [ ]:
path_to_repo = Path("..").resolve()
path_to_data = path_to_repo / "data"

In [ ]:
df_transformed = pd.read_feather(path_to_data / "transformed")

df, y, nas = process_df(df_transformed, "SalePrice")

In [ ]:
print(df.shape)
df.head()

In [ ]:
len(y)
y

In [ ]:
model = RandomForestRegressor(n_estimators=10, n_jobs=-1, random_state=42)

%time model.fit(df, y)

In [ ]:
x = df.iloc[0]
y_true = np.exp(y[0])
x, y_true

In [ ]:
x = [x]  # wrap the data point into a list
y_pred = model.predict(x)  # prediction, as list
y_pred = y_pred[0]  # take first element of predicted list
y_pred = np.exp(y_pred)  # exponentialize
y_pred

In [ ]:
model.score(df, y)

In statistics, the coefficient of determination, denoted R2 or r2 and pronounced "R squared", is the proportion of the variance in the dependent variable that is predictable from the independent variable(s). https://en.wikipedia.org/wiki/Coefficient_of_determination

Wow, an r^2 of 0.98 - that's great, right? Well, perhaps not...

Possibly **the most important idea** in machine learning is that of having separate training & validation data sets. As motivation, suppose you don't divide up your data, but instead use all of it.  And suppose you have lots of parameters:

<img src="../docs/imgs/notebooks/underfitting_overfitting.png" alt="" style="width: 70%"/>
<center>
[Underfitting and Overfitting](https://datascience.stackexchange.com/questions/361/when-is-a-model-underfitted)
</center>

The error for the pictured data points is lowest for the model on the far right (the blue curve passes through the red points almost perfectly), yet it's not the best choice.  Why is that?  If you were to gather some new data points, they most likely would not be on that curve in the graph on the right, but would be closer to the curve in the middle graph.

This illustrates how using all our data can lead to **overfitting**. A validation set helps diagnose this problem.

In [ ]:
n_total = len(df)
n_valid = 12000  # same as Kaggle's test set size
n_train = n_total - n_valid
n_small = 20000

print("full number of data points : {}".format(n_total))
print("number of validation data points : {}".format(n_valid))
print("number of training data points : {}".format(n_train))
print("number of subsampled training points : {}".format(n_small))

In [ ]:
X_train, X_valid = split_vals(df, n_train)
y_train, y_valid = split_vals(y, n_train)

X_small, _ = split_vals(X_train, n_small)
y_small, _ = split_vals(y_train, n_small)

print("Number of small training data points: X = {}, y = {}".format(X_small.shape, y_small.shape))
print("Number of full training data points: X = {}, y = {}".format(X_train.shape, y_train.shape))
print("Number of validation data points: X = {}, y = {}".format(X_valid.shape, y_valid.shape))

### $\bullet$ Base model

[Back to top](#plan)

Let's try our model again, this time with separate training and validation sets.

In [ ]:
def rmse(y_gold, y_pred):
    return math.sqrt(((y_gold - y_pred) ** 2).mean())

In [ ]:
def print_score(m, X_train, y_train, X_valid, y_valid):
    print("RMSE on train set: {:.4f}".format(rmse(m.predict(X_train), y_train)))
    print("RMSE on valid set: {:.4f}".format(rmse(m.predict(X_valid), y_valid)))
    print("R^2 on train set: {:.4f}".format(m.score(X_train, y_train)))
    print("R^2 on valid set: {:.4f}".format(m.score(X_valid, y_valid)))
    if hasattr(m, "oob_score_"):
        print("R^2 on oob set: {:.4f}".format(m.oob_score_))
    return

In [ ]:
# model on small training data
base_model = RandomForestRegressor(n_estimators=10, n_jobs=-1, random_state=42)

%time base_model.fit(X_small, y_small)
print_score(base_model, X_small, y_small, X_valid, y_valid)

In [ ]:
# model on full training data
base_model = RandomForestRegressor(n_estimators=10, n_jobs=-1, random_state=42)

%time base_model.fit(X_train, y_train)
print_score(base_model, X_train, y_train, X_valid, y_valid)

An r^2 in the high-80's isn't bad at all (and the RMSLE puts us around rank 100 of 470 on the Kaggle leaderboard), but we can see from the validation set score that we're over-fitting badly. To understand this issue, let's simplify things down to a single small tree.

### $\bullet$ Decision Tree Regressor

[Back to top](#plan)


Let us describe the process of construction of a single decision tree, over a dataset $\mathcal{D}_1$ described by two explanatory variables $X_1$ and $X_2$ and an output variable $Y$ :<br>

| $X^1$ | $X^2$ | $Y$ |
|-----|-----|-----|
| $x^1_1$ | $x^2_1$ | $y_1$ |
| $x^1_2$ | $x^2_2$ | $y_2$ |
| $x^1_3$ | $x^2_3$ | $y_3$ |



#### 1. Splitting process

The decision tree will first split the dataset $\mathcal{D}_1$ into two sub-datasets $\mathcal{D}_2$ and $\mathcal{D}_3$, by choosing a variable among the explanatory variables (let's say $X^1$) and a separation threshold $d_1$ for this variable. The way of selecting a variable and a threshold is detailed in the *Splitting Strategy* paragraph below.<br> Such choice divides the dataset $\mathcal{D}_1$ into <br>

$$\mathcal{D}_2 = \displaystyle \lbrace  (x^1, x^2, y) \in \mathcal{D}_1 \, | \, x^1 \leqslant d_1 \rbrace \qquad \qquad \mathcal{D}_3 = \{ (x^1, x^2, y) \in \mathcal{D}_1 \, | \, x^1 > d_1 \}$$

<img src="../docs/imgs/notebooks/decision_tree_1_.png" alt="" style="width: 50%"/>

If the tree construction would stop here, the resulting decision tree $\mathcal{T}$ is a prediction model that associates, to any new data point $(o^1, o^2)$, the value accoring to the following two cases : if $o^1 \leqslant d_1$, then $\mathcal{T}(o^1, o^2)$ is the average values of the output values among the $\mathcal{D}_2$ subset, whereas if $o^1 > d_1$, then $\mathcal{T}(o^1, o^2)$ is the average values of the output values among the $\mathcal{D}_3$ subset.<br>
This is more formally written as :

$$ \mathcal{T}(o^1, o^2) = 
\begin{cases}
\displaystyle \frac{1}{\vert \mathcal{D}_2 \vert } \sum _{(x^1, x^2, y) \, \in \, \mathcal{D}_2} y & \text{if } o^1 \leqslant d_1 \\
\displaystyle \frac{1}{\vert \mathcal{D}_3 \vert} \sum _{(x^1, x^2, y) \, \in \, \mathcal{D}_3} y & \text{if } o^1 > d_1\\
\end{cases}
$$


Actually the tree does not stop here in general, but implement subsequent splittings as detained in the *Recurrence process* paragraph below.


#### 2. Splitting Strategy

The splitting strategy depends on the choice of a measure of *Heterogeneity* of a set of output values. For decision tree regression, the heterogeneity of a set $Y = \{ y_1, y_2, y_3, ...\}$ of real values (i.e. floating numbers) is taken as its variance $Var(Y)$ :

$$ \overline{Y} = \frac{1}{\vert Y \vert}\displaystyle \sum _{y\, \in \, Y} y \qquad \qquad Var(Y) = \frac{1}{\vert Y \vert}\displaystyle \sum _{y\, \in \, Y} ( y - \overline{Y} ) ^2$$


It is a heterogeneity measure as the more the set $Y$ is concentrated around a single value (e.g. its average) then the less the variance is.<br>

Now that the homogeneity of a set has been defined, the splitting strategy consists in chosing, given a set of data $\mathcal{D}$, a variable $X^*$ among $X^1$ and $X^2$ as well as a threshold $d^*$ that minimizes, among the possible variables and thresholds, the summed heterogeneity



$$ \mathcal{D}_{left} = \lbrace (x^1, x^2, y) \in \mathcal{D}, \;  x^i \leqslant d \rbrace $$
$$ \mathcal{D}_{right} = \lbrace (x^1, x^2, y) \in \mathcal{D}, \;  x^i > d \rbrace $$

<br>
<br>

$$
H(\mathcal{D}, X^i, d) =  \vert \mathcal{D}_{left} \vert \times Var(\lbrace y \, | \, (x^1, x^2, y) \in \mathcal{D}_{left} \rbrace) \quad +  \quad \vert \mathcal{D}_{right} \vert \times Var(\lbrace y \, | \, (x^1, x^2, y) \in \mathcal{D}_{right} \rbrace)
$$


This quantity is the variance which is obtained by splitting the dataset along the $X^*$ variable about the threshold $d^*$, and the resulting best choice is the one that minimizes this variance.


#### 3. Recurrence process

The full decision tree is built by _iterating_ the splitting process : If the splitting strategy yields a first decision along a variable $X^1$ with threshold $d_1$, yielding two sub-datasets $\mathcal{D}_2$ and $\mathcal{D}_3$, then the same splitting strategy is performed on $\mathcal{D}_2$ and $\mathcal{D}_3$ independently. The process stops when the splitting strategy does not find any candidate variable and threshold over any of the sub-datasets obtained so far.<br>

As an illustration, let us suppose that, after the first splitting step descrived earlier, that the splitting strategy finds no splitting on $\mathcal{D}_3$ and a splitting of $\mathcal{D}_2$ according to the variable $X^2$ and threshold $d_2$, yielding 

$$\mathcal{D}_4 = \displaystyle \lbrace  (x^1, x^2, y) \in \mathcal{D}_2 \, | \, x^2 \leqslant d_2 \rbrace \qquad \qquad \mathcal{D}_5 = \{ (x^1, x^2, y) \in \mathcal{D}_2 \, | \, x^2 > d_2 \}$$

Suppose moreover that a third step yields a splitting of $\mathcal{D}_5$ according to the variable $X^1$ and threshold $d_3$, yielding 

$$\mathcal{D}_6 = \displaystyle \lbrace  (x^1, x^2, y) \in \mathcal{D}_5 \, | \, x^1 \leqslant d_3 \rbrace \qquad \qquad \mathcal{D}_7 = \{ (x^1, x^2, y) \in \mathcal{D}_5 \, | \, x^1 > d_3 \}$$

Then the resulting decision tree can be pictured as follows :

<img src="../docs/imgs/notebooks/decision_tree_3.png" alt="" style="width: 50%"/>


The resulting prediction model $\mathcal{T}$ is then given by the following function on a new data point $(o^1, o^2)$ :

$$ \mathcal{T}(o^1, o^2) = 
\begin{cases}
\displaystyle \frac{1}{\vert \mathcal{D}_3 \vert} \sum _{(x^1, x^2, y) \, \in \, \mathcal{D}_3} y & \text{if } o^1 > d_1\\
\displaystyle \frac{1}{\vert \mathcal{D}_4 \vert} \sum _{(x^1, x^2, y) \, \in \, \mathcal{D}_4} y & \text{if } o^1 \leqslant d_1 \text{ and } o^2 \leqslant d_2 \\
\displaystyle \frac{1}{\vert \mathcal{D}_6 \vert} \sum _{(x^1, x^2, y) \, \in \, \mathcal{D}_6} y & \text{if } o^1 \leqslant d_1 \text{ and } o^2 > d_2 \text{ and } o^1 \leqslant d_3 \\
\displaystyle \frac{1}{\vert \mathcal{D}_7 \vert} \sum _{(x^1, x^2, y) \, \in \, \mathcal{D}_7} y & \text{if } o^1 \leqslant d_1 \text{ and } o^2 > d_2 \text{ and } o^1 > d_3 \\
\end{cases}
$$


#### Final remarks 

- A single splitting step consists in finding the best splits *for each subsample where splitting is possible*. 
- When looking at an optimal choice of splitting, *all variables are systematically used* even if they were selected in a revious splitting.
- The splitting of a subsample stops when it contains a single data point, or when a certain *stopping criterion has been reached*, which prevents any subsequent splitting.
- The final tree prediction values always belong to a finite set of possible values, one value per leaf, given as the averaged ouput of training data points belonging to the leaf.

**Option 1**: Using _sklearn.tree.DecisionTreeRegressor_

In [ ]:
help(DecisionTreeRegressor)

In [ ]:
model_dt = DecisionTreeRegressor(
    criterion="squared_error",
    splitter="best",
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=42,
)

In [ ]:
%time model_dt.fit(X_small, y_small)
print_score(model_dt, X_small, y_small, X_valid, y_valid)

In [ ]:
model_dt = DecisionTreeRegressor(
    criterion="squared_error",
    splitter="best",
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=42,
)

In [ ]:
%time model_dt.fit(X_small, y_small)
print_score(model_dt, X_small, y_small, X_valid, y_valid)

In [ ]:
model_dt = DecisionTreeRegressor(
    criterion="squared_error",
    splitter="best",
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0001,
    ccp_alpha=0.0,
    random_state=42,
)

In [ ]:
%time model_dt.fit(X_small, y_small)
print_score(model_dt, X_small, y_small, X_valid, y_valid)

In [ ]:
model_dt = DecisionTreeRegressor(
    criterion="squared_error",
    splitter="best",
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=42,
)

In [ ]:
%time model_dt.fit(X_small, y_small)
print_score(model_dt, X_small, y_small, X_valid, y_valid)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=1, figsize=(15, 10), dpi=300)
plot_tree(model_dt, filled=True)

In [ ]:
draw_tree(model_dt, feature_names=X_small.columns.tolist(), precision=3)

**Option 2**: Using _sklearn.ensemble.RandomForestRegressor_

In [ ]:
model_rf = RandomForestRegressor(
    # parameters shared with DecisionTreeRegressor
    n_estimators=1,  # = the model is a single tree
    criterion="squared_error",
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=42,
    # RandomForestRegressor specific hyperparameters
    bootstrap=False,  # default = True
    oob_score=False,
    max_samples=None,
    # extra parameters
    warm_start=False,
    n_jobs=-1,
    verbose=0,
)

In [ ]:
%time model_rf.fit(X_small, y_small)
print_score(model_rf, X_small, y_small, X_valid, y_valid)

In [ ]:
# first install graphviz source files, see:
#

draw_tree(model_rf.estimators_[0], X_small.columns.tolist(), precision=3)

Using a Random Forest with 2 trees :

In [ ]:
model_rf = RandomForestRegressor(
    # parameters shared with DecisionTreeRegressor
    n_estimators=2,  # 1 = the model is a single tree
    criterion="squared_error",
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=0.5,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=42,
    # RandomForestRegressor specific hyperparameters
    bootstrap=True,  # default = True
    oob_score=False,
    max_samples=0.5,
    # extra parameters
    warm_start=False,
    n_jobs=-1,  # None
    verbose=0,
)

In [ ]:
%time model_rf.fit(X_small, y_small)
print_score(model_rf, X_small, y_small, X_valid, y_valid)

In [ ]:
draw_tree(model_rf.estimators_[0], X_small.columns.tolist(), precision=3)

In [ ]:
X_small.Enclosure

In [ ]:
df_transformed.loc[:10, "Enclosure"]

In [ ]:
draw_tree(model_rf.estimators_[1], X_small.columns.tolist(), precision=3)

In [ ]:
X_small.fiProductClassDesc

In [ ]:
df_transformed.loc[:50, "fiProductClassDesc"]

Let's see what happens if we create a bigger tree.

In [ ]:
model_dt = DecisionTreeRegressor(
    criterion="squared_error",  # 'mse',
    splitter="best",
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    random_state=42,
    ccp_alpha=0.0,
)

In [ ]:
model_dt.fit(X_small, y_small)
print_score(model_dt, X_small, y_small, X_valid, y_valid)

Let us remove the constrain on the tree depth :

In [ ]:
model_dt = DecisionTreeRegressor(
    criterion="squared_error",  # 'mse',
    splitter="best",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    random_state=42,
    ccp_alpha=0.0,
)

In [ ]:
model_dt.fit(X_small, y_small)
print_score(model_dt, X_small, y_small, X_valid, y_valid)

### $\bullet$ Tree bagging

[Back to top](#plan)

The training set result looks great! But the validation set is worse than our original model. This is why we need to use *bagging* of multiple trees to get more generalizable results. To learn about bagging in random forests, let's start with our basic model again.<br>

By the way, Bagging is the reduction of **B**oostrap **Agg**regation.

In [ ]:
help(RandomForestRegressor)

In [ ]:
base_model = RandomForestRegressor(
    # parameters shared with DecisionTreeRegressor
    n_estimators=10,  # 100 # 1 = the model is a single tree
    criterion="squared_error",
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=42,
    # RandomForestRegressor specific hyperparameters
    bootstrap=True,
    oob_score=False,
    max_samples=None,
    # extra parameters
    warm_start=False,
    n_jobs=-1,
    verbose=0,
)

%time base_model.fit(X_small, y_small)
print_score(base_model, X_small, y_small, X_valid, y_valid)

In [ ]:
point = X_valid.iloc[0, :]
print(point)

price = y_valid[0]
print(price, np.exp(price))

In [ ]:
# the model makes prediction with model.predict method
# base_model.predict

# predict takes a list as input, so we turn our single point as a list (with a single element)
base_model.predict([point])

# the model returns a list, so we take the first element, and apply exponetial
np.exp(base_model.predict([point])[0])

We'll grab the predictions for each individual tree, and look at one example.

In [ ]:
# model prediction on single data point
trees = base_model
point = [X_valid.iloc[0, :]]
pred = trees.predict(point)

print("Type:", type(pred))
print("Shape:", pred.shape)
print("Value:", np.exp(pred[0]))

In [ ]:
# first tree prediction on single data point
tree = base_model.estimators_[0]  # chose between 0 and 9 since we have 10 estimators
point = [X_valid.iloc[0, :]]  # tree.predict takes an iterable of data points as input
pred = tree.predict(point)  # tree.predict returns a numpy tensor of predictions as output

print("Type:", type(pred))
print("Shape:", pred.shape)
print("Value:", np.exp(pred[0]))

In [ ]:
# first tree prediction on single data point
tree = base_model.estimators_[1]  # chose between 0 and 9 since we have 10 estimators
point = [X_valid.iloc[0, :]]  # tree.predict takes an iterable of data points as input
pred = tree.predict(point)  # tree.predict returns a numpy tensor of predictions as output

print("Type:", type(pred))
print("Shape:", pred.shape)
print("Value:", np.exp(pred[0]))

In [ ]:
# all tree predictions on single data point
trees = base_model.estimators_
point = [X_valid.iloc[0, :]]
preds = np.asarray([tree.predict(point) for tree in trees])

print("Type:", type(preds))
print("Shape:", preds.shape)
print("Value:", np.exp(preds))

In [ ]:
# all tree predictions on all data
preds = np.stack([tree.predict(X_valid) for tree in base_model.estimators_])  # or np.asarray()

# predictions of the 10 trees, mean, and actual value (for first point in validation set)
preds[:, 0], base_model.predict(point)[0], np.mean(preds[:, 0]), y_valid[0]

In [ ]:
plt.plot([metrics.r2_score(y_valid, np.mean(preds[:num_trees], axis=0)) for num_trees in range(1, 11)])

The shape of this curve suggests that adding more trees isn't going to help us much. Let's check. (Compare this to our original model on a sample)

In [ ]:
model = RandomForestRegressor(
    # parameters shared with DecisionTreeRegressor
    n_estimators=20,  # 100 # 1 = the model is a single tree
    criterion="squared_error",
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=42,
    # RandomForestRegressor specific hyperparameters
    bootstrap=True,
    oob_score=False,
    max_samples=None,
    # extra parameters
    warm_start=False,
    n_jobs=-1,
    verbose=0,
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
# all tree predictions on all data
preds = np.stack([tree.predict(X_valid) for tree in model.estimators_])  # or np.asarray()

# predictions of the 10 trees, mean, and actual value (for first point in validation set)
preds.shape, preds[:, 0], np.mean(preds[:, 0]), y_valid[0]

plt.plot([metrics.r2_score(y_valid, np.mean(preds[:num_trees], axis=0)) for num_trees in range(1, 21)])

In [ ]:
model = RandomForestRegressor(
    # parameters shared with DecisionTreeRegressor
    n_estimators=30,  # 100 # 1 = the model is a single tree
    criterion="squared_error",
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=42,
    # RandomForestRegressor specific hyperparameters
    bootstrap=True,
    oob_score=False,
    max_samples=None,
    # extra parameters
    warm_start=False,
    n_jobs=-1,
    verbose=0,
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
# all tree predictions on all data
preds = np.stack([tree.predict(X_valid) for tree in model.estimators_])  # or np.asarray()

# predictions of the 10 trees, mean, and actual value (for first point in validation set)
preds.shape, preds[:, 0], np.mean(preds[:, 0]), y_valid[0]

plt.plot([metrics.r2_score(y_valid, np.mean(preds[:num_trees], axis=0)) for num_trees in range(1, 31)])

In [ ]:
model = RandomForestRegressor(
    # parameters shared with DecisionTreeRegressor
    n_estimators=50,  # 100 # 1 = the model is a single tree
    criterion="squared_error",
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=42,
    # RandomForestRegressor specific hyperparameters
    bootstrap=True,
    oob_score=False,
    max_samples=None,
    # extra parameters
    warm_start=False,
    n_jobs=-1,
    verbose=0,
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
# all tree predictions on all data
preds = np.stack([tree.predict(X_valid) for tree in model.estimators_])  # or np.asarray()

# predictions of the 10 trees, mean, and actual value (for first point in validation set)
preds.shape, preds[:, 0], np.mean(preds[:, 0]), y_valid[0]

plt.plot([metrics.r2_score(y_valid, np.mean(preds[:num_trees], axis=0)) for num_trees in range(1, 51)])

Let us try this time on the full training set :

In [ ]:
model = RandomForestRegressor(
    # parameters shared with DecisionTreeRegressor
    n_estimators=50,  # 100 # 1 = the model is a single tree
    criterion="squared_error",
    max_depth=10,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=42,
    # RandomForestRegressor specific hyperparameters
    bootstrap=True,
    oob_score=False,
    max_samples=None,
    # extra parameters
    warm_start=False,
    n_jobs=-1,
    verbose=0,
)

%time model.fit(X_train, y_train)
print_score(model, X_train, y_train, X_valid, y_valid)

In [ ]:
# all tree predictions on all data
preds = np.stack([tree.predict(X_valid) for tree in model.estimators_])  # or np.asarray()

# predictions of the 10 trees, mean, and actual value (for first point in validation set)
preds.shape, preds[:, 0], np.mean(preds[:, 0]), y_valid[0]

plt.plot([metrics.r2_score(y_valid, np.mean(preds[:num_trees], axis=0)) for num_trees in range(1, 51)])

### $\bullet$ Out-of-bag (OOB) score

[Back to top](#plan)

Is our validation set worse than our training set because we're over-fitting, or because the validation set is for a different time period, or a bit of both? With the existing information we've shown, we can't tell. However, random forests have a very clever trick called *out-of-bag (OOB) error* which can handle this (and more!)

The idea is to calculate error on the training set, but only include the trees in the calculation of a row's error where that row was *not* included in training that tree. This allows us to see whether the model is over-fitting, without needing a separate validation set.

This also has the benefit of allowing us to see whether our model generalizes, even if we only have a small amount of data so want to avoid separating some out to create a validation set.

This is as simple as adding one more parameter to our model constructor. We print the OOB error last in our `print_score` function below.

In [ ]:
model = RandomForestRegressor(
    n_estimators=40,
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False,
    max_samples=None,
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

Bagging trick :

| -| Tree 1  | Tree 2 |
|-----|-----|-----|
|Data 1 | (2 times) | - |
|Data 2  | (1 time) |(2 times) |
|Data 3 | - |(1 time) |

$$
R^2 \text{ on train set } > R^2 \text{ on oob set } \implies \text{ The model is overfitting} \\ 
R^2 \text{ on oob set } > R^2 \text{ on valid set } \implies \text{ Training / Validation sets carry differences} \\
$$

In [ ]:
model = RandomForestRegressor(
    n_estimators=40,
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False,
    max_samples=None,
)

%time model.fit(X_train, y_train)
print_score(model, X_train, y_train, X_valid, y_valid)

### $\bullet$ Stoping criteria

[Back to top](#plan)

A way to reduce over-fitting is to grow our trees less deeply. We do this by specifying (with `min_samples_leaf` and `min_samples_split`) that we require some minimum number of rows in every leaf node. This has two benefits:

- There are less decision rules for each leaf node; simpler models should generalize better
- The predictions are made by averaging more rows in the leaf node, resulting in less volatility

In [ ]:
def decision_tree_max_depth(tree):
    children_left = tree.children_left
    children_right = tree.children_right

    def walk(node_id):
        if children_left[node_id] != children_right[node_id]:
            left_max = 1 + walk(children_left[node_id])
            right_max = 1 + walk(children_right[node_id])
            return max(left_max, right_max)
        else:  # leaf
            return 1

    root_node_id = 0
    return walk(root_node_id)

Baseline model

In [ ]:
model = RandomForestRegressor(
    n_estimators=40,
    criterion="squared_error",
    # stoping criteria
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    # bagging parameters
    max_features=None,
    bootstrap=True,
    oob_score=True,
    max_samples=None,
    # other
    warm_start=False,
    n_jobs=-1,
    random_state=42,
    verbose=0,
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
tree = model.estimators_[0].tree_

decision_tree_max_depth(tree)

In [ ]:
for i, tree in enumerate(model.estimators_):
    print("tree {} has depth {}".format(i, decision_tree_max_depth(tree.tree_)))

In [ ]:
model = RandomForestRegressor(
    n_estimators=40,
    min_samples_split=15,
    min_samples_leaf=15,
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False,
    max_samples=None,
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
tree = model.estimators_[0].tree_

decision_tree_max_depth(tree)

In [ ]:
for i, tree in enumerate(model.estimators_):
    print("tree {} has depth {}".format(i, decision_tree_max_depth(tree.tree_)))

### $\bullet$ Data subsampling

[Back to top](#plan)

It turns out that one of the easiest ways to avoid over-fitting is also one of the best ways to speed up analysis: *subsampling*.<br>

**Let's return to using our full dataset**, so that we can demonstrate the impact of this technique.<br>

The basic idea is this: rather than limit the total amount of data that our model can access, let's instead limit it to a *different* random subset per tree. That way, given enough trees, the model can still see *all* the data, but for each individual tree it'll be just as fast as if we had cut down our dataset as before.

In [ ]:
model = RandomForestRegressor(
    n_estimators=40,
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=None,
)

%time model.fit(X_train, y_train)
print_score(model, X_train, y_train, X_valid, y_valid)

In [ ]:
model = RandomForestRegressor(
    n_estimators=40,
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=0.15,  # default = None
)

%time model.fit(X_train, y_train)
print_score(model, X_train, y_train, X_valid, y_valid)

In [ ]:
model = RandomForestRegressor(
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=0.15,  # default = None
)

%time model.fit(X_train, y_train)
print_score(model, X_train, y_train, X_valid, y_valid)

In [ ]:
model = RandomForestRegressor(
    n_estimators=40,
    criterion="squared_error",
    # stoping criteria
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    # bagging parameters
    max_features=None,
    bootstrap=True,
    oob_score=True,
    max_samples=0.5,
    # other
    warm_start=False,
    n_jobs=-1,
    random_state=42,
    verbose=0,
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

### $\bullet$ Feature subsampling

[Back to top](#plan)

We can also increase the amount of variation amongst the trees by not only use a sample of rows for each tree, but to also using a sample of *columns* for each *split*. We do this by specifying `max_features`, which is the proportion of features to randomly select from at each split.

- None
- 0.5
- 'sqrt'

In [ ]:
model = RandomForestRegressor(
    n_estimators=40,
    # max_features = 0.5, # default = None
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=None,
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
model = RandomForestRegressor(
    n_estimators=40,
    max_features=0.5,  # default = None
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=None,
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
model = RandomForestRegressor(
    n_estimators=40,
    max_features=0.5,  # default = None
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=None,
)

%time model.fit(X_train, y_train)
print_score(model, X_train, y_train, X_valid, y_valid)

Let us plot the oob error as a function of the number of trees, for some feature sampling euristics (see [sklearn documentation](http://scikit-learn.org/stable/auto_examples/ensemble/plot_ensemble_oob.html))

<img src="../docs/imgs/notebooks/oob_error_rates.png" alt="" style="width: 100%"/>

In [ ]:
logging.info("warn")

# NOTE: Setting the `warm_start` construction parameter to `True` disables
# support for parallelized ensembles but is necessary for tracking the OOB
# error trajectory during training.
ensemble_clfs = [
    (
        "RandomForestRegressor, max_features = 'log2'",
        RandomForestRegressor(warm_start=True, oob_score=True, max_features="log2", random_state=42),
    ),
    (
        "RandomForestRegressor, max_features = 'sqrt'",
        RandomForestRegressor(warm_start=True, oob_score=True, max_features="sqrt", random_state=42),
    ),
    (
        "RandomForestRegressor, max_features = 0.25",
        RandomForestRegressor(warm_start=True, oob_score=True, max_features=0.25, random_state=42),
    ),
    (
        "RandomForestRegressor, max_features = 0.5",
        RandomForestRegressor(warm_start=True, oob_score=True, max_features=0.5, random_state=42),
    ),
    (
        "RandomForestRegressor, max_features = None",
        RandomForestRegressor(warm_start=True, oob_score=True, max_features=None, random_state=42),
    ),
]

# Map a classifier name to a list of (<n_estimators>, <error rate>) pairs.
error_rate = OrderedDict((label, []) for label, _ in ensemble_clfs)

# Range of `n_estimators` values to explore.
min_estimators = 10
max_estimators = 150

for label, clf in ensemble_clfs:
    for i in range(min_estimators, max_estimators + 1):
        clf.set_params(n_estimators=i)
        clf.fit(X_small, y_small)

        # Record the OOB error for each `n_estimators=i` setting.
        oob_error = 1 - clf.oob_score_
        error_rate[label].append((i, oob_error))

# Generate the "OOB error rate" vs. "n_estimators" plot.
plt.figure(figsize=(15, 5))
for label, clf_err in error_rate.items():
    xs, ys = zip(*clf_err)
    plt.plot(xs, ys, label=label)

# Show the plot
plt.xlim(min_estimators, max_estimators)
plt.ylim(0.0, 0.2)
plt.xlabel("n_estimators")
plt.ylabel("OOB error rate")
plt.legend(loc="upper right")
plt.show()

In [ ]:
model = RandomForestRegressor(
    n_estimators=40,
    criterion="squared_error",
    # stoping criteria
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    # bagging parameters
    max_features=None,
    bootstrap=True,
    oob_score=True,
    max_samples=None,
    # other
    warm_start=False,
    n_jobs=-1,
    random_state=42,
    verbose=0,
)

%time model.fit(X_train, y_train)
print_score(model, X_train, y_train, X_valid, y_valid)

In [ ]:
model = RandomForestRegressor(
    n_estimators=40,
    criterion="squared_error",
    # stoping criteria
    max_depth=15,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    # bagging parameters
    max_features=0.5,
    bootstrap=True,
    oob_score=True,
    max_samples=None,
    # other
    warm_start=False,
    n_jobs=-1,
    random_state=42,
    verbose=0,
)

%time model.fit(X_train, y_train)
print_score(model, X_train, y_train, X_valid, y_valid)

#### Combining all together

In [ ]:
model = RandomForestRegressor(
    n_estimators=100,
    min_samples_split=10,
    max_features=0.5,  # default = None
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=0.75,  # default = None
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
model = RandomForestRegressor(
    n_estimators=100,
    min_samples_split=15,
    max_features=0.75,  # default = None
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=0.75,  # default = None
)

%time model.fit(X_train, y_train)
print_score(model, X_train, y_train, X_valid, y_valid)

In [ ]:
model = RandomForestRegressor(
    n_estimators=150,
    min_samples_split=10,
    max_features=0.5,  # default = None
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=0.75,  # default = None
)

%time model.fit(X_train, y_train)
print_score(model, X_train, y_train, X_valid, y_valid)

### $\bullet$ Bagging as general Ensemble method

[Back to top](#plan)

The Bagging (Bootstrap Aggregation) strategy does not need to be applied to trees only, it can be applied to many different base models. This is done using the `BaggingRegressor` meta-model.<br>

A model generated with `RandomForestRegressor` is almost identical to a model instanciated from `BaggingRegressor`over a base `DecisionTreeRegressor`, with two differences:
- It has its own implementation, making it faster for training
- The feature sampling is _different at every tree node_ (when used)

The 2nd aspect does not hold for models instanciated with `BaggingRegressor`, where feature sampling is done _once and for all_ for each tree at instanciation.

In [ ]:
# native Random Forest
model = RandomForestRegressor(
    n_estimators=100,
    min_samples_split=2,
    max_features=0.5,  # default = None
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=None,
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
# Random Forest built using the Bagging meta-model
base_model = DecisionTreeRegressor(
    criterion="squared_error",
    splitter="best",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    ccp_alpha=0.0,
    random_state=42,
)

model = BaggingRegressor(
    estimator=base_model,
    n_estimators=100,
    max_features=0.5,
    max_samples=1.0,
    bootstrap=True,
    bootstrap_features=False,
    oob_score=True,
    warm_start=False,
    n_jobs=-1,
    random_state=42,
    verbose=0,
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor

In [ ]:
# Bagged model using Linear Regression as base model
base_model = LinearRegression()

model = BaggingRegressor(
    estimator=base_model,
    n_estimators=100,
    max_features=0.5,
    max_samples=1.0,
    bootstrap=True,
    bootstrap_features=False,
    oob_score=True,
    warm_start=False,
    n_jobs=-1,
    random_state=42,
    verbose=0,
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

In [ ]:
help(MLPRegressor)

In [ ]:
# Bagged model using Linear Regression as base model
base_model = MLPRegressor(hidden_layer_sizes=(100, 50), activation="tanh")

model = BaggingRegressor(
    estimator=base_model,
    n_estimators=10,
    max_features=0.5,
    max_samples=1.0,
    bootstrap=True,
    bootstrap_features=False,
    oob_score=True,
    warm_start=False,
    n_jobs=-1,
    random_state=42,
    verbose=0,
)

%time model.fit(X_small, y_small)
print_score(model, X_small, y_small, X_valid, y_valid)

### $\bullet$ Cross validation

[Back to top](#plan)

Random Forests, and more generally Bagging models, come natively with the computation of an out-of-bag score. Non-ensemblist methods does not come with such score, and an alternative is to consider a **Cross validation score**. This score can be computationally expensive, but does not waste too much data (as is the case when fixing an arbitrary validation set).

In $k$-fold CV, the training set is split into $k$ smaller sets (by default $k$ consecutive evenly-sized splits, with $k=5$). The following procedure is followed for each of the $k$ “folds”:

- A model is trained using $k-1$ of the folds as training data;
- the resulting model is validated on the remaining part of the data (i.e., it is used as a test set to compute the model's `score` method).

The performance measure reported by $k$-fold cross-validation is then the average of the values computed in the loop. 


<img src="../docs/imgs/notebooks/cross_valid.png" alt="" style="width: 40%"/>
<center>
Cross validation
</center>

See the [scikitlearn documentation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation)

This score is dependant on some parameters, such as the number $k$ of folds. Cross validation is also useful for Random Forests when performing **hyperparameter tuning**, addressed in next paragraph.

In [ ]:
# native Random Forest
model = RandomForestRegressor(
    n_estimators=100,
    min_samples_split=2,
    max_features=None,
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=None,
)

In [ ]:
# default is cv = 5
scores = cross_val_score(model, X_small, y_small, cv=8, n_jobs=-1)

print(scores)
print("r2 is {:.3f} with a standard deviation of {:.3f}".format(scores.mean(), scores.std()))

- Cross validation with a randomized splitting strategy.<br>
This strategy is recommended when the data does not follow a particular order, but *should be avoided for ordered data* (ex data with a temporal ordering).

In [ ]:
help(model.score)

In [ ]:
# randomized splitting strategy
cv = KFold(n_splits=8, shuffle=True, random_state=42)

scores = cross_val_score(model, X_small, y_small, cv=cv, n_jobs=-1)

print(scores)
print("r2 is {:.3f} with a standard deviation of {:.3f}".format(scores.mean(), scores.std()))

- Find the best model among min_samples_split = 2, 10, 20 according to 5-fold cross validation
- Find the best model among max_depth = 3, 10, 20, 40 according to 5-fold cross validation

In [ ]:
# native Random Forest
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=3,
    min_samples_split=2,
    max_features=None,
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=None,
)


# randomized splitting strategy
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X_small, y_small, cv=cv, n_jobs=-1)
print(scores)
print("r2 is {:.3f} with a standard deviation of {:.3f}".format(scores.mean(), scores.std()))

In [ ]:
# native Random Forest
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=2,
    max_features=None,
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=None,
)


# randomized splitting strategy
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X_small, y_small, cv=cv, n_jobs=-1)
print(scores)
print("r2 is {:.3f} with a standard deviation of {:.3f}".format(scores.mean(), scores.std()))

In [ ]:
# native Random Forest
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=2,
    max_features=None,
    n_jobs=-1,
    random_state=42,
    bootstrap=True,
    oob_score=True,  # default = False
    max_samples=None,
)


# randomized splitting strategy
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X_small, y_small, cv=cv, n_jobs=-1)
print(scores)
print("r2 is {:.3f} with a standard deviation of {:.3f}".format(scores.mean(), scores.std()))

- Change the score formula (ex: switch from $R^2$ to Root Mean Squared Error)

In [ ]:
scores = cross_val_score(model, X_small, y_small, cv=8, scoring="r2", n_jobs=-1)

print(scores)
print("r2 is {:.3f} with a standard deviation of {:.3f}".format(scores.mean(), scores.std()))

In [ ]:
scores = cross_val_score(model, X_small, y_small, cv=8, scoring="neg_root_mean_squared_error", n_jobs=-1)

print(scores)
print("RMSE is {:.3f} with a standard deviation of {:.3f}".format(-scores.mean(), scores.std()))

### $\bullet$ Hyperparameter tuning for final model selection

[Back to top](#plan)

Hyper-parameters are parameters that are not directly learnt by calling the model's `fit` method. These hyperparameters are usually set manually, and each different choice will lead to a possibly different result when computing the model's score on a test set. It is however important **not to finetune these hyperparameters on results obtained on the test set**: They must instead be selected using either a dedicated validation set, or through cross validation. 

<img src="../docs/imgs/notebooks/hyperparameter_tuning.png" alt="" style="width: 40%"/>
<center>
$\textit{Place of Hyperparameter tuning in the process of building a prediction model}$
</center>


<br>
It is completely ok for a Data Scientist to calibrate a model's hyperparameters manually on a separate validation set, as it also informs on the most impacting ones. To complete this approach, we describe here automatic hyperparameter finetuning with cross validation.

Concretely, a hyperparameter search consists of:

- an estimator (classifier or regressor such as `RandomForestRegressor`)

- a parameter space

- a method for searching or sampling candidate parameters

- a cross-validation scheme

- a score function

Two generic approaches to parameter search are provided in scikit-learn: for given values, `GridSearchCV` exhaustively considers all parameter combinations, while `RandomizedSearchCV` can sample a given number of candidates from a parameter space with a specified distribution. Both these tools have successive halving counterparts `HalvingGridSearchCV` and `HalvingRandomSearchCV`, which can be much faster at finding a good parameter combination.



See the [scikitlearn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html#sklearn.model_selection.GridSearchCV)

In [ ]:
model.get_params()

In [ ]:
# several grids can be cast into a list
# here we provide only one grid
param_grid = [
    {
        "n_estimators": [5, 10],
        "max_depth": [10, 20, None],
        "min_samples_split": [2, 10],
        "max_features": [0.4, 0.75, 1.0],
        "max_samples": [0.4, 0.75, 1.0],
    },
]

model = RandomForestRegressor(
    random_state=42,
    bootstrap=True,
)

In [ ]:
tuned_model = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring=None,  # uses estimator's default score method
    n_jobs=-1,
    refit=True,  # keep a fitted version of the overall best model
    cv=5,
    return_train_score=True,
    verbose=2,
)

In [ ]:
tuned_model.fit(X_small, y_small)

In [ ]:
best_model = tuned_model.best_estimator_
best_model

In [ ]:
print_score(best_model, X_small, y_small, X_valid, y_valid)

In [ ]:
# randomized splitting strategy
cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(best_model, X_small, y_small, cv=cv, n_jobs=-1)

print(scores)
print("r2 is {:.3f} with a standard deviation of {:.3f}".format(scores.mean(), scores.std()))

### $\bullet$ Final model export

[Back to top](#plan)

In [ ]:
# save model on disk
with open(path_to_repo / "checkpoints" / "RF_regressor.pk", "wb") as file:
    dill.dump(best_model, file)

[Back to top](#plan)